In [ ]:
!pip install -q transformers datasets accelerate seqeval

In [ ]:
import pandas as pd
import numpy as np
import torch
import re

from datasets import Dataset, load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)

In [ ]:
DATA_PATH = "/kaggle/input/competitions/kaz-punct-hackathon/"

train_df = pd.read_csv(DATA_PATH + "train_example.csv")
test_df = pd.read_csv(DATA_PATH + "test.csv")

print("train:", len(train_df), "test:", len(test_df))

In [ ]:
def strip_and_label(text):

    tokens = text.split()

    labels = []
    clean = []

    for t in tokens:

        if t.endswith("?"):
            labels.append("QUESTION")
            clean.append(t[:-1].lower())

        elif t.endswith((".", "!")):
            labels.append("PERIOD")
            clean.append(t[:-1].lower())

        elif t.endswith(","):
            labels.append("COMMA")
            clean.append(t[:-1].lower())

        else:
            labels.append("O")
            clean.append(t.lower())

    return " ".join(clean), " ".join(labels)

In [ ]:
dataset_stream = load_dataset(
    "kz-transformers/multidomain-kazakh-dataset",
    split="train",
    streaming=True
)

In [ ]:
rows = []

for i,row in enumerate(dataset_stream):

    text = row["text"]

    sentences = re.split(r"[.!?]", text)

    for s in sentences:

        if s.strip():

            clean, labels = strip_and_label(s.strip())

            if len(clean.split()) == len(labels.split()):

                rows.append({
                    "input_text": clean,
                    "labels": labels
                })

    if len(rows) > 40000:
        break

extra_df = pd.DataFrame(rows)

print("extra sentences:", len(extra_df))

In [ ]:
train_full = pd.concat(
    [train_df[["input_text","labels"]], extra_df],
    ignore_index=True
)

print("total training:", len(train_full))

In [ ]:
LABELS = ["O","COMMA","PERIOD","QUESTION"]

label2id = {l:i for i,l in enumerate(LABELS)}
id2label = {i:l for l,i in label2id.items()}

In [ ]:
def convert_row(row):

    tokens = row["input_text"].split()
    labels = row["labels"].split()

    return {
        "tokens": tokens,
        "ner_tags": [label2id[l] for l in labels]
    }

dataset = Dataset.from_list([
    convert_row(r) for _,r in train_full.iterrows()
])

dataset = dataset.train_test_split(test_size=0.05)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")

In [ ]:
def tokenize(example):

    tokenized = tokenizer(
        example["tokens"],
        is_split_into_words=True,
        truncation=True
    )

    word_ids = tokenized.word_ids()

    labels = []
    prev = None

    for word_id in word_ids:

        if word_id is None:
            labels.append(-100)

        elif word_id != prev:
            labels.append(example["ner_tags"][word_id])

        else:
            labels.append(-100)

        prev = word_id

    tokenized["labels"] = labels

    return tokenized


dataset = dataset.map(tokenize)

In [ ]:
model = AutoModelForTokenClassification.from_pretrained(
    "xlm-roberta-base",
    num_labels=4,
    id2label=id2label,
    label2id=label2id
)

In [ ]:
training_args = TrainingArguments(

    output_dir="./model",

    learning_rate=2e-5,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    num_train_epochs=1,

    logging_steps=200,

    save_strategy="no",

    report_to="none"
)

In [ ]:
data_collator = DataCollatorForTokenClassification(tokenizer)

trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=dataset["train"],

    eval_dataset=dataset["test"],

    data_collator=data_collator
)

In [ ]:
trainer.train()

In [ ]:
def predict(text):

    tokens = text.split()

    encoded = tokenizer(
        tokens,
        is_split_into_words=True,
        return_tensors="pt"
    )

    encoded = {k:v.to(model.device) for k,v in encoded.items()}

    with torch.no_grad():
        outputs = model(**encoded)

    preds = outputs.logits.argmax(-1)[0].cpu().numpy()

    word_ids = tokenizer(tokens,is_split_into_words=True).word_ids()

    labels=[]
    prev=None

    for i,w in enumerate(word_ids):

        if w is None:
            continue

        if w!=prev:
            labels.append(id2label[preds[i]])

        prev=w

    return labels

In [ ]:
question_words = {"ма","ме","ба","бе","па","пе","қалай","қайда","қашан","неге"}

comma_words = {"бірақ","алайда","сондықтан","өйткені","яғни","демек"}

rows=[]

for _,row in test_df.iterrows():

    tokens=row["input_text"].split()

    preds=predict(row["input_text"])

    if len(preds)<len(tokens):
        preds+=["O"]*(len(tokens)-len(preds))

    if len(preds)>len(tokens):
        preds=preds[:len(tokens)]

    preds[-1]="PERIOD"

    if tokens[-1] in question_words:
        preds[-1]="QUESTION"

    for i in range(len(tokens)-1):
        if tokens[i+1] in comma_words:
            preds[i]="COMMA"

    rows.append({
        "id":row["id"],
        "labels":" ".join(preds)
    })

submission=pd.DataFrame(rows)

submission.to_csv("submission_big_model.csv",index=False)

submission.head()

In [ ]:
print(submission.iloc[2]["labels"])